In [10]:
using Pkg

Pkg.add("DataFrames")
Pkg.activate("Data_Assimilation")
Pkg.develop(path="../Krylov.jl")   # change path to local Krylov fork
Pkg.develop(path="../JSOSolvers.jl") 

   Resolving package versions...
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Project.toml`
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Manifest.toml`
  Activating project at `~/Desktop/DataAssim.jl/Data_Assimilation`
   Resolving package versions...
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Project.toml`
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Project.toml`
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Manifest.toml`


In [11]:
Pkg.add("ADNLPModels")

   Resolving package versions...
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Project.toml`
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Manifest.toml`


In [12]:

# Inclure tes fichiers locaux
include("../DataAssim.jl/src/lorenz95.jl")
include("../DataAssim.jl/src/operators.jl")

   Resolving package versions...
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Project.toml`
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Manifest.toml`


build_model (generic function with 1 method)

In [13]:
using NLPModels, ADNLPModels
using ForwardDiff
using StaticArrays
using JSOSolvers

function nonlinear_funcval(x, y, obs, xb, B, R)
    eo = y .- gop(obs, x)
    eb = x .- xb
    fx = eltype(x).(1/2) * eb'* invdot(B, eb) + eltype(x).(1/2) * eo'*invdot(R, eo)
    return fx
end

nonlinear_funcval (generic function with 1 method)

In [ ]:
n = 10000
nt = 8
dt = 0.025
F = 8.0
rng = MersenneTwister(1234)

sigmaR =  0.1
total_space_obs = 500
total_time_obs = 2   # m = total_space_obs*total_time_obs
sigmaB =  0.8

# Model
model = Lorenz95Model(F, dt)

# Observations

space_inds_obs = round.(Int, range(1, n; length=total_space_obs))
time_inds_obs = round.(Int, range(1, nt-1; length=total_time_obs))
m = total_space_obs*total_time_obs
obs = ObsOperator(sigmaR, space_inds_obs, n, time_inds_obs, nt, m, model)
R = RMatrix(sigmaR)

# Background
B = BMatrix(sigmaB, n)

# Spin-up
xt = 3.0 .* ones(n) .+ randn(rng, n)
xt = traj(model, xt, 5000)

# Background state
xb = xt .+ randn(rng, n) .* sigmaB

# Observations
y = generate_obs(obs, xt)

# Construct ADNLPModel
f(x) = nonlinear_funcval(x, y, obs, xb, B, R)

f (generic function with 1 method)

In [15]:
using DataFrames
function run_solver(nlp, subsolver; memory=nothing)

    if memory === nothing
        stats = trunk(nlp,
            max_time=10000.0,
            max_iter=500,
            verbose=0,
            subsolver=subsolver
        )
    else
        stats = trunk(nlp,
            max_time=10000.0,
            max_iter=500,
            verbose=0,
            subsolver=subsolver,
            subsolver_kwargs=(memory=memory,)
        )
    end

    row = (
        solver = string(subsolver) * (memory === nothing ? "" : "_m$(memory)"),
        status = stats.status,
        norm_sol = norm(stats.solution),
        objective = stats.objective,
        iter = stats.iter,
        obj_eval = nlp.counters.neval_obj,
        grad_eval = nlp.counters.neval_grad,
        hprod = nlp.counters.neval_hprod,
        time = stats.elapsed_time
    )

    reset!(nlp)

    return row
end

run_solver (generic function with 1 method)

In [16]:

x0 = copy(xb)

nlp = ADNLPModel(f, x0, backend = :optimized)


ADNLPModel - Model with automatic differentiation backend ADModelBackend{
  ReverseDiffADGradient,
  ReverseDiffADHvprod,
  EmptyADbackend,
  EmptyADbackend,
  EmptyADbackend,
  SparseReverseADHessian,
  EmptyADbackend,
}
  Problem name: Generic
   All variables: ████████████████████ 10000  All constraints: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
            free: ████████████████████ 10000             free: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           lower: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                lower: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           upper: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                upper: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
         low/upp: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0              low/upp: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           fixed: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                fixed: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
          infeas: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0               infeas: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
            nnzh: ( 98.32% sparsity)   840500          linear: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
                          

In [17]:
results = DataFrame()

push!(results, run_solver(nlp, :cg))
push!(results, run_solver(nlp, :lbfgs; memory=100))
push!(results, run_solver(nlp, :diom; memory=100))
push!(results, run_solver(nlp, :lbfgs; memory=50))
push!(results, run_solver(nlp, :diom; memory=50))

Row,solver,status,norm_sol,objective,iter,obj_eval,grad_eval,hprod,time
,String,Symbol,Float64,Float64,Int64,Int64,Int64,Int64,Float64
1,cg,first_order,436.805,781.556,54,212,53,772,130.667
2,lbfgs_m100,first_order,436.805,781.556,65,306,62,802,128.421
3,diom_m100,first_order,436.781,783.204,56,251,55,759,121.461
4,lbfgs_m50,first_order,436.805,781.556,65,306,62,804,128.417
5,diom_m50,first_order,436.781,783.204,56,251,55,761,122.838


In [18]:
using PrettyTables
pretty_table(results)

┌────────────┬─────────────┬──────────┬───────────┬───────┬──────────┬──────────
│     solver │      status │ norm_sol │ objective │  iter │ obj_eval │ grad_ev ⋯
│     String │      Symbol │  Float64 │   Float64 │ Int64 │    Int64 │     Int ⋯
├────────────┼─────────────┼──────────┼───────────┼───────┼──────────┼──────────
│         cg │ first_order │  436.805 │   781.556 │    54 │      212 │         ⋯
│ lbfgs_m100 │ first_order │  436.805 │   781.556 │    65 │      306 │         ⋯
│  diom_m100 │ first_order │  436.781 │   783.204 │    56 │      251 │         ⋯
│  lbfgs_m50 │ first_order │  436.805 │   781.556 │    65 │      306 │         ⋯
│   diom_m50 │ first_order │  436.781 │   783.204 │    56 │      251 │         ⋯
└────────────┴─────────────┴──────────┴───────────┴───────┴──────────┴──────────
                                                               3 columns omitted
